# Lighting-direction split walkthrough

A face-recognition model should be tested on lighting directions absent from training. This notebook demonstrates the directional embedding and checks the fold grouping on synthetic metadata. It does not use or display real face images.


In [ ]:
import numpy as np
import pandas as pd
from src.splits import spherical_coords_to_unit_vectors, generate_spherical_bins

angles = [(-100, 0), (-80, 10), (-50, 0), (-30, 10), (0, 0), (10, 10), (40, 0), (60, 10), (90, 0), (110, 10)]
rows = [dict(person=f'person_{subject}', path=f'image_{subject}_{index}.pgm', pose='P00', A=azimuth, E=elevation) for subject in range(3) for index, (azimuth, elevation) in enumerate(angles)]
metadata = pd.DataFrame(rows)
vectors = spherical_coords_to_unit_vectors(metadata['A'].to_numpy(), metadata['E'].to_numpy())
assert np.allclose(np.linalg.norm(vectors, axis=1), 1)
assigned, _ = generate_spherical_bins(metadata, n_bins=4, seed=42)
print(assigned.groupby('bin_id').size())


The implementation uses **Euclidean k-means on three-dimensional unit vectors**, followed by best-effort sample balancing. Whole lighting directions remain together. The exact bin counts depend on the supplied dataset; inspect the generated fold summary rather than assuming every bin is equal.


In [ ]:
for held_out_bin in sorted(assigned['bin_id'].unique()):
    train = assigned[assigned['bin_id'] != held_out_bin]
    test = assigned[assigned['bin_id'] == held_out_bin]
    assert set(train['bin_id']).isdisjoint(test['bin_id'])
print('All outer folds isolate their held-out lighting bin.')
